# poolguard Quickstart

This notebook walks through a complete patient-pool workflow:

1. Synthetic multi-site cohort with intentional covariate shift
2. Reference cohort registration
3. Shift detection (SMD, KS, MMD)
4. Per-pool model evaluation with bootstrap CIs
5. Cross-pool heterogeneity (Cochran's Q, I²)
6. Transportability (IPW, g-computation)
7. Regulator-ready exports and audit verification

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from poolguard import PoolGuard, PoolGuardConfig

## 1. Build a synthetic multi-site dataset

Site B is shifted older on average to mimic a real deployment drift.

In [ ]:
rng = np.random.default_rng(42)
n_per_site = 150

site_a = pd.DataFrame(
    {
        "age": rng.normal(60, 8, n_per_site),
        "biomarker": rng.normal(0, 1, n_per_site),
    }
)
site_b = pd.DataFrame(
    {
        "age": rng.normal(68, 8, n_per_site),  # intentional shift
        "biomarker": rng.normal(0.4, 1, n_per_site),
    }
)
site_c = pd.DataFrame(
    {
        "age": rng.normal(62, 8, n_per_site),
        "biomarker": rng.normal(0.1, 1, n_per_site),
    }
)

X = pd.concat([site_a, site_b, site_c], ignore_index=True)
y = (X["age"] > 62).astype(int)
pools = pd.Series(["site_a"] * n_per_site + ["site_b"] * n_per_site + ["site_c"] * n_per_site)

print(X.groupby(pools)["age"].agg(["mean", "std"]))

## 2. Fit a model and configure PoolGuard

In [ ]:
model = LogisticRegression(max_iter=500, random_state=0).fit(X, y)

config = PoolGuardConfig(
    bootstrap_n=500,
    random_state=42,
    smd_threshold=0.1,
    metadata={"protocol": "DEMO-001", "model": "logistic_regression"},
)
pg = PoolGuard(model=model, config=config, study_id="quickstart-demo")
ref = pg.fit_reference(X, y, pools, reference_pool="site_a")
print(f"Reference pool: {ref}")

## 3. Population shift detection

In [ ]:
shift = pg.detect_shift(X, pools)
print(shift.summary())
print("Flagged pools:", shift.flagged_pools)

## 4. Per-pool evaluation

In [ ]:
evaluation = pg.evaluate(X, y, pools)
print(evaluation.summary())

## 5. Heterogeneity testing

In [ ]:
heterogeneity = pg.test_heterogeneity(X, y, pools, metric="auc")
print(heterogeneity.summary())

## 6. Transportability assessment

In [ ]:
transport = pg.assess_transportability(X, y, pools, metric="auc")
print(transport.summary())

## 7. Full workflow and exports

In [ ]:
analysis = pg.run_full_analysis(X, y, pools, metric="auc")

# JSON/HTML artifacts (paths relative to notebook directory)
analysis.evaluation.to_json("evaluation_report.json")
analysis.shift.to_html("shift_report.html")

print(f"Audit events: {len(pg.audit)}")
print(f"Chain verified: {pg.audit.verify()}")
print(f"Head hash: {pg.audit.head_hash()[:16]}...")

## Optional: pool column in a single DataFrame

When the pool label is already a column, pass `pool_column` to avoid separate `pools` arrays.

In [ ]:
df = X.copy()
df["site"] = pools
pg2 = PoolGuard(model=model, pool_column="site", config=config)
pg2.fit_reference(df, y)
pg2.detect_shift(df)  # pools inferred from column